# Optativa III: Ciencia de Datos
## Semana 6 — Ingeniería de datos con fuentes VIVAS: pipelines desde APIs online

**Universidad de Especialidades UNE · Plantel Centro**
**Ingeniería en Computación · 9.º semestre · Semana 6 · Sesiones 1 y 2**

---

### Propósito

En la actividad anterior construiste pipelines desde un CSV **fijo**. Pero en el mundo real, los datos **cambian todo el tiempo**: los precios de una tienda suben y bajan, el stock se agota, aparecen productos nuevos. En esta actividad tu pipeline consumirá **datos vivos** de bases de datos **online que se actualizan**:

- Una **tienda online** (API de productos tipo e-commerce) que devuelve precios, stock y categorías.
- Una **base de datos colaborativa real** (Open Food Facts) que crece cada minuto con aportes de usuarios de todo el mundo.

Aprenderás:
- **Arquitecturas de datos:** pipelines, ETL vs ELT, batch vs streaming, warehouse/lake/lakehouse.
- **Consumir una API online** que cambia entre una consulta y otra.
- **Modelado dimensional:** esquema estrella/copo de nieve, normalización vs desnormalización.
- Construir un **ETL completo** que extrae de la web, transforma y carga a SQLite.

La actividad tiene dos partes:

- **PARTE A — Ejemplo guiado por el profesor.** Construiremos juntos un pipeline que extrae de una API online, la modela y la carga a SQLite.
- **PARTE B — Tu turno.** Consumirás una base viva, detectarás que sus datos cambian, y construirás tu propio pipeline.

Responderás **50 preguntas** modificando código, diseñando arquitecturas e interpretando datos que cambian.

### Requisito

Solo necesitas Google Colab con internet (para consultar las APIs). No hay que subir ningún archivo: los datos llegan de la web.

### Cómo trabajar

Ejecuta cada celda con **Shift + Enter** en orden, responde las preguntas, completa los retos y, al final, guarda una copia en Drive y entrégala en Classroom.

---
# PARTE A — Conceptos de arquitectura de datos

## La analogía del agua

Los datos son como agua:
- **Pipeline (tubería):** el sistema que mueve el agua de la fuente al destino.
- **ETL / ELT:** ¿purificas el agua ANTES o DESPUÉS de almacenarla?
- **Batch vs streaming:** ¿llenas cubetas cada hora (batch) o el agua fluye continuamente (streaming)?
- **Warehouse / lake / lakehouse:** ¿la guardas embotellada y ordenada, o en un lago crudo?

In [1]:
print("""
ETL (Extract - Transform - Load)  ← el clásico
  1. EXTRAER de la fuente   2. TRANSFORMAR (limpiar) ANTES de guardar   3. CARGAR ya limpio
  Uso: data warehouses. Los datos llegan listos.

ELT (Extract - Load - Transform)  ← el moderno
  1. EXTRAER   2. CARGAR crudo al destino (data lake)   3. TRANSFORMAR cuando se necesite
  Uso: big data, almacenamiento barato.
""")


ETL (Extract - Transform - Load)  ← el clásico
  1. EXTRAER de la fuente   2. TRANSFORMAR (limpiar) ANTES de guardar   3. CARGAR ya limpio
  Uso: data warehouses. Los datos llegan listos.

ELT (Extract - Load - Transform)  ← el moderno
  1. EXTRAER   2. CARGAR crudo al destino (data lake)   3. TRANSFORMAR cuando se necesite
  Uso: big data, almacenamiento barato.



> **1.** ¿Cuál es la diferencia clave entre ETL y ELT? (Pista: el ORDEN de la transformación.)

**Respuesta:** En ETL se transforma ANTES de cargar (al destino solo llega dato limpio); en ELT se carga crudo y se transforma DESPUÉS, dentro del destino, cuando se necesita.

> **2.** Una tienda online quiere que sus reportes de ventas SIEMPRE lleguen con datos limpios y validados. ¿ETL o ELT? Justifica.

**Respuesta:** ETL. Si los reportes siempre deben llegar limpios y validados, la limpieza tiene que pasar antes de cargar; así nadie consulta datos sin validar.

> **3.** Si quisieras guardar TODOS los datos crudos de la tienda (por si acaso) porque el almacenamiento es barato, ¿ETL o ELT?

**Respuesta:** ELT: se carga todo crudo (es barato) y se transforma solo lo que se vaya necesitando.

In [2]:
print("""
BATCH (por lotes): procesa datos JUNTOS en intervalos. Ej: total de ventas del día, a medianoche.
STREAMING (continuo): procesa cada dato AL LLEGAR. Ej: actualizar el stock en cuanto alguien compra.
""")


BATCH (por lotes): procesa datos JUNTOS en intervalos. Ej: total de ventas del día, a medianoche.
STREAMING (continuo): procesa cada dato AL LLEGAR. Ej: actualizar el stock en cuanto alguien compra.



> **4.** Para cada caso de una tienda online, decide BATCH o STREAMING y justifica:
> (a) Calcular las ventas totales del mes. (b) Descontar el stock en cuanto un cliente compra. (c) Enviar un reporte de inventario cada noche. (d) Mostrar cuántas personas ven un producto ahora mismo.

**Respuesta:** (a) Batch: es un cálculo de cierre, basta con correrlo al terminar el mes. (b) Streaming: el stock debe bajar en el momento o se venden productos agotados. (c) Batch: tiene horario fijo, cada noche. (d) Streaming: "ahora mismo" exige procesar cada visita al llegar.

> **5.** Cuando consultas una API de productos que se actualiza, ¿estás haciendo un procesamiento más cercano a batch o a streaming? Argumenta (pista: ¿pides todo de una vez o dato por dato en tiempo real?).

**Respuesta:** Más cercano a batch: en cada consulta pido un lote (50 productos) de una sola vez, no recibo cada cambio en tiempo real. Si la consultara cada pocos segundos se acercaría a streaming (micro-batch).

In [3]:
print("""
DATA WAREHOUSE: datos estructurados, en tablas, listos para consultar con SQL. (Biblioteca ordenada.)
DATA LAKE: datos crudos de cualquier tipo (JSON, texto, imágenes). Barato y flexible. (Un lago.)
LAKEHOUSE: híbrido; la flexibilidad del lake + la estructura del warehouse.
""")


DATA WAREHOUSE: datos estructurados, en tablas, listos para consultar con SQL. (Biblioteca ordenada.)
DATA LAKE: datos crudos de cualquier tipo (JSON, texto, imágenes). Barato y flexible. (Un lago.)
LAKEHOUSE: híbrido; la flexibilidad del lake + la estructura del warehouse.



> **6.** Ordena warehouse, lake y lakehouse del MÁS al MENOS estructurado.

**Respuesta:** Warehouse (más estructurado) → lakehouse → lake (menos estructurado).

> **7.** Los datos de una API llegan en formato JSON (crudo, anidado). Si los guardaras TAL CUAL, sin procesar, ¿en qué tipo de almacenamiento estarían: warehouse o lake?

**Respuesta:** En un data lake: datos crudos guardados en su formato original.

> **8.** En esta actividad, tras limpiar los datos de la API los cargaremos a SQLite (tablas estructuradas). ¿A cuál de los tres se parece SQLite? Justifica.

**Respuesta:** A un data warehouse en miniatura: guarda tablas estructuradas y ya limpias, con esquema definido, y se consulta con SQL.

### Reto 1 — Diseña un pipeline para una tienda online (en papel)

Una tienda online quiere analizar sus ventas. Sus datos vienen de: (1) el catálogo de productos (una API que cambia varias veces al día), (2) las compras (llegan en tiempo real), y (3) reseñas de clientes (texto). Diseña el pipeline en 4-6 pasos.

> **9.** Escribe tu diseño del pipeline. Especifica: ¿ETL o ELT? ¿batch, streaming o ambos? ¿warehouse, lake o lakehouse? Justifica cada decisión.

**Respuesta:**

1. Extraer el catálogo de la API cada hora (batch).
2. Recibir las compras como eventos en tiempo real (streaming) y descontar el stock al momento.
3. Guardar las reseñas crudas (texto) una vez al día.
4. Limpiar y unir catálogo + compras en un esquema estrella (fact_ventas + dim_producto, dim_tiempo, dim_cliente).
5. Procesar las reseñas (por ejemplo, sentimiento) y ligarlas al producto.
6. Dashboards y reportes sobre el esquema estrella.

**ELT**, porque hay texto y JSON que conviene guardar crudos antes de saber cómo se van a usar. **Batch y streaming**: el catálogo y las reseñas aguantan unas horas de retraso, las compras no. **Lakehouse**, porque mezcla datos estructurados (ventas) con no estructurados (reseñas).

> **10.** ¿Por qué el catálogo (que cambia varias veces al día) NO se puede tratar como un archivo fijo? ¿Cada cuánto tendrías que volver a extraerlo?

**Respuesta:** Porque un archivo fijo es una foto vieja: en pocas horas sus precios y su stock ya no coinciden con la tienda. Habría que volver a extraerlo al ritmo con que cambia, por ejemplo cada hora.

---
# PARTE A (cont.) — Pipeline ETL desde una API ONLINE

Ahora construimos un pipeline que extrae datos de una **tienda online real** vía API. Los datos llegan por internet, en formato JSON.

## Paso 0 — Preparación

In [4]:
import numpy as np
import pandas as pd
import sqlite3
import requests
from datetime import datetime

print("Librerías listas. Vamos a consultar datos en vivo de la web.")

Librerías listas. Vamos a consultar datos en vivo de la web.


## Paso 1 — EXTRACT desde la API online

Usaremos **DummyJSON**, una API pública gratuita que simula una tienda online (productos, precios, stock, categorías). Se consulta por internet, sin registro.

In [5]:
# EXTRACT: pedir productos a la tienda online
url = "https://dummyjson.com/products?limit=50"

# Un pipeline robusto maneja el caso de que la API falle (sin internet, caída, etc.)
try:
    respuesta = requests.get(url, timeout=15)
    respuesta.raise_for_status()
    datos = respuesta.json()
    productos = datos["products"]
    print("EXTRACT en vivo desde la tienda online. Productos recibidos:", len(productos))
    print("Total de productos en la tienda:", datos.get("total"))
except Exception as e:
    print("La API no respondió (", type(e).__name__, "). Usando datos de respaldo.")
    productos = [
        {"id":1,"title":"iPhone 9","price":549,"discountPercentage":12.96,"rating":4.69,"stock":94,"brand":"Apple","category":"smartphones"},
        {"id":2,"title":"iPhone X","price":899,"discountPercentage":17.94,"rating":4.44,"stock":34,"brand":"Apple","category":"smartphones"},
        {"id":3,"title":"Samsung Universe 9","price":1249,"discountPercentage":15.46,"rating":4.09,"stock":36,"brand":"Samsung","category":"smartphones"},
        {"id":4,"title":"MacBook Pro","price":1749,"discountPercentage":11.02,"rating":4.57,"stock":83,"brand":"Apple","category":"laptops"},
        {"id":5,"title":"Samsung Galaxy Book","price":1499,"discountPercentage":4.15,"rating":4.25,"stock":50,"brand":"Samsung","category":"laptops"},
        {"id":6,"title":"Lipstick","price":20,"discountPercentage":8.71,"rating":4.36,"stock":68,"brand":"Chanel","category":"fragrances"},
    ]

print("\nEjemplo de un producto (JSON crudo):")
print(productos[0])

EXTRACT en vivo desde la tienda online. Productos recibidos: 50
Total de productos en la tienda: 194

Ejemplo de un producto (JSON crudo):
{'id': 1, 'title': 'Essence Mascara Lash Princess', 'description': 'The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.', 'category': 'beauty', 'price': 9.99, 'discountPercentage': 10.48, 'rating': 2.56, 'stock': 99, 'tags': ['beauty', 'mascara'], 'brand': 'Essence', 'sku': 'BEA-ESS-ESS-001', 'weight': 4, 'dimensions': {'width': 15.14, 'height': 13.08, 'depth': 22.99}, 'warrantyInformation': '1 week warranty', 'shippingInformation': 'Ships in 3-5 business days', 'availabilityStatus': 'In Stock', 'reviews': [{'rating': 3, 'comment': 'Would not recommend!', 'date': '2025-04-30T09:41:02.053Z', 'reviewerName': 'Eleanor Collins', 'reviewerEmail': 'eleanor.collins@x.dummyjson.com'}, {'rating': 4, 'comment': 'Very satisfied!', 'date

> **11.** ¿Cuántos productos extrajiste de la API? ¿En qué se diferencia extraer de una API (por internet) vs leer un archivo CSV local?

**Respuesta:** 50 productos (la tienda tiene 194 en total). De una API los datos llegan por internet en JSON, en el momento en que se piden; pueden cambiar entre consultas y la petición puede fallar. Un CSV local es fijo y siempre está disponible.

> **12.** El pipeline tiene un `try/except`. ¿Por qué un pipeline profesional debe prepararse para que la fuente online FALLE? ¿Qué pasa con el resto del programa si no se maneja el error?

**Respuesta:** Porque una fuente online puede estar caída, lenta o sin internet, y no la controlamos. Sin el try/except la excepción detiene el programa y ninguna celda posterior corre. En esta sesión Open Food Facts falló varias veces con HTTPError, así que no es un caso raro.

> **13.** Un producto llega como un diccionario JSON con muchos campos. ¿Qué campos ves que serían útiles para un análisis de ventas?

**Respuesta:** price, discountPercentage, rating, stock, category y brand; también availabilityStatus, minimumOrderQuantity y reviews.

## Paso 2 — TRANSFORM

Convertimos el JSON en una tabla, limpiamos y creamos variables útiles.

In [6]:
# TRANSFORM: JSON -> tabla, seleccionar columnas, crear features
df = pd.DataFrame(productos)

# Nos quedamos con las columnas útiles (algunas APIs traen decenas)
columnas = ["id", "title", "brand", "category", "price", "discountPercentage", "rating", "stock"]
df = df[[c for c in columnas if c in df.columns]].copy()

# Feature derivada: precio final con el descuento aplicado
df["precio_final"] = (df["price"] * (1 - df["discountPercentage"]/100)).round(2)

# Feature: nivel de stock
df["nivel_stock"] = pd.cut(df["stock"], bins=[-1, 20, 60, 9999],
                           labels=["Bajo", "Medio", "Alto"])

# Marca de tiempo de la extracción (importante con datos vivos)
df["fecha_extraccion"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Datos transformados:", df.shape)
print(df[["title","brand","category","price","precio_final","stock","nivel_stock"]].head())

Datos transformados: (50, 11)
                           title           brand category  price  \
0  Essence Mascara Lash Princess         Essence   beauty   9.99   
1  Eyeshadow Palette with Mirror  Glamour Beauty   beauty  19.99   
2                Powder Canister    Velvet Touch   beauty  14.99   
3                   Red Lipstick  Chic Cosmetics   beauty  12.99   
4                Red Nail Polish    Nail Couture   beauty   8.99   

   precio_final  stock nivel_stock  
0          8.94     99        Alto  
1         16.35     34       Medio  
2         13.51     89        Alto  
3         11.41     91        Alto  
4          7.96     79        Alto  


> **14.** ¿Qué hace `pd.DataFrame(productos)` con la lista de diccionarios JSON? ¿Por qué convertir el JSON en una tabla facilita el análisis?

**Respuesta:** Convierte cada diccionario en una fila y cada clave en una columna. En tabla se puede filtrar, agrupar, crear columnas y cargar a SQL directamente; con el JSON anidado habría que recorrerlo a mano.

> **15.** Creamos la columna `fecha_extraccion`. ¿Por qué es CRÍTICO registrar CUÁNDO se extrajeron los datos cuando la fuente se actualiza constantemente?

**Respuesta:** Porque el mismo análisis da resultados distintos según el momento. Sin la fecha no se sabe a qué versión de los datos corresponde un número ni se pueden comparar extracciones.

> **16.** La feature `precio_final` aplica el descuento. ¿Por qué es más útil para un análisis de ventas que el precio de lista?

**Respuesta:** Porque es lo que realmente paga el cliente; los ingresos y las comparaciones de precio salen mal si se usa el precio de lista.

### Reto 2 — Añade una feature

Crea una columna `es_oferta` que valga 1 si `discountPercentage` es mayor a 15, y 0 si no. ¿Cuántos productos están en oferta fuerte?

In [7]:
# RETO 2: es_oferta = 1 si el descuento es mayor a 15%
df["es_oferta"] = (df["discountPercentage"] > 15).astype(int)
print("Productos en oferta fuerte:", df["es_oferta"].sum(), "de", len(df))

Productos en oferta fuerte: 6 de 50


> **17.** ¿Cuántos productos tienen oferta fuerte (>15% de descuento)? Como los datos son vivos, ¿este número podría cambiar si ejecutas mañana?

**Respuesta:** 6 de 50. Sí podría cambiar: si la tienda ajusta sus descuentos, mañana el conteo sería otro.

## Paso 3 — Modelado dimensional (esquema estrella)

Diseñamos cómo se guardarán los datos: una tabla de **hechos** (métricas) rodeada de **dimensiones** (descriptores).

In [8]:
print("""
ESQUEMA ESTRELLA para la tienda:

        dim_categoria            dim_marca
              |                      |
              +---- FACT_PRODUCTOS --+
                    (precio, rating, stock...)

- FACT_PRODUCTOS: métricas numéricas + claves (IDs).
- dim_categoria, dim_marca: descriptores (texto).
""")

# Dimensión categoría
dim_cat = pd.DataFrame({"category": df["category"].unique()})
dim_cat["cat_id"] = range(1, len(dim_cat)+1)

# Dimensión marca
dim_marca = pd.DataFrame({"brand": df["brand"].dropna().unique()})
dim_marca["marca_id"] = range(1, len(dim_marca)+1)

print("Dimensión categoría:")
print(dim_cat)
print("\nDimensión marca:")
print(dim_marca.head())


ESQUEMA ESTRELLA para la tienda:

        dim_categoria            dim_marca
              |                      |
              +---- FACT_PRODUCTOS --+
                    (precio, rating, stock...)

- FACT_PRODUCTOS: métricas numéricas + claves (IDs).
- dim_categoria, dim_marca: descriptores (texto).

Dimensión categoría:
              category  cat_id
0               beauty       1
1           fragrances       2
2            furniture       3
3            groceries       4
4      home-decoration       5
5  kitchen-accessories       6

Dimensión marca:
            brand  marca_id
0         Essence         1
1  Glamour Beauty         2
2    Velvet Touch         3
3  Chic Cosmetics         4
4    Nail Couture         5


In [9]:
# Tabla de HECHOS: reemplazamos texto por IDs y guardamos métricas
fact = (df.merge(dim_cat, on="category")
          .merge(dim_marca, on="brand", how="left"))
fact_productos = fact[["id", "cat_id", "marca_id", "price", "precio_final",
                       "rating", "stock"]].copy()
print("Tabla de hechos:", fact_productos.shape)
fact_productos.head()

Tabla de hechos: (50, 7)


,id,cat_id,marca_id,price,precio_final,rating,stock
0,1,1,1.0,9.99,8.94,2.56,99
1,2,1,2.0,19.99,16.35,2.86,34
2,3,1,3.0,14.99,13.51,4.64,89
3,4,1,4.0,12.99,11.41,4.36,91
4,5,1,5.0,8.99,7.96,4.32,79


> **18.** ¿Qué diferencia hay entre la tabla de HECHOS y las de DIMENSIONES? ¿Qué guarda cada una?

**Respuesta:** La de hechos guarda las medidas numéricas (price, precio_final, rating, stock) y las claves hacia las dimensiones. Las dimensiones guardan los descriptores de texto para filtrar y agrupar (nombre de categoría, marca).

> **19.** En la tabla de hechos, la categoría y la marca se reemplazan por IDs numéricos. ¿Por qué guardar un ID en lugar de repetir "smartphones" o "Apple" en miles de filas?

**Respuesta:** Un entero ocupa menos que un texto repetido y el JOIN por número es más rápido. Además, si el nombre cambia se corrige en una sola fila de la dimensión y no hay inconsistencias como "Apple" vs "apple".

> **20.** El esquema estrella está desnormalizado para analítica. ¿Por qué esto acelera las consultas de un dashboard, aunque duplique algo de información?

**Respuesta:** Porque toda la información queda a un solo JOIN de distancia: las dimensiones son tablas planas y pequeñas, así que las consultas del dashboard son simples y rápidas. Repetir algo de texto en las dimensiones cuesta poco comparado con la velocidad que se gana.

## Paso 4 — LOAD a SQLite y consulta

In [10]:
# LOAD: cargar el modelo completo a SQLite
conn = sqlite3.connect("tienda_online.db")
dim_cat.to_sql("dim_categoria", conn, if_exists="replace", index=False)
dim_marca.to_sql("dim_marca", conn, if_exists="replace", index=False)
fact_productos.to_sql("fact_productos", conn, if_exists="replace", index=False)

# Consulta analítica: precio promedio y stock por categoría
query = """
SELECT c.category AS categoria,
       COUNT(*) AS num_productos,
       ROUND(AVG(f.price), 0) AS precio_promedio,
       SUM(f.stock) AS stock_total
FROM fact_productos f
JOIN dim_categoria c ON f.cat_id = c.cat_id
GROUP BY c.category
ORDER BY num_productos DESC
"""
print(pd.read_sql(query, conn).to_string(index=False))
conn.close()

          categoria  num_productos  precio_promedio  stock_total
          groceries             27              5.0         1417
    home-decoration              5             39.0          220
          furniture              5           1200.0          245
         fragrances              5             84.0          280
             beauty              5             13.0          392
kitchen-accessories              3              8.0          185


> **21.** ¿Qué hace el `JOIN` entre la tabla de hechos y la dimensión categoría? ¿Por qué es necesario para "reconstruir" el nombre de la categoría?

**Respuesta:** Une cada fila de hechos con su fila de dim_categoria por cat_id. Así se recupera el texto ("groceries") a partir del número para poder agrupar y mostrar el nombre.

> **22.** ¿Cuál categoría tiene más productos y cuál más stock total? (Ejecuta y reporta; recuerda que con datos vivos puede variar.)

**Respuesta:** groceries tiene más productos (27) y también más stock total (1,417 unidades).

> **23.** Acabas de construir un ETL completo desde una fuente ONLINE: Extract (API) → Transform (limpieza + features) → Load (SQLite). Resume en 3 frases qué pasó en cada fase.

**Respuesta:** Extract: pedimos 50 productos en JSON a la API de DummyJSON. Transform: los pasamos a tabla, elegimos columnas y creamos precio_final, nivel_stock y fecha_extraccion. Load: guardamos las dimensiones y la tabla de hechos en tienda_online.db y la consultamos con SQL.

### Reto 3 — Consulta por marca

Escribe una consulta SQL que muestre el rating promedio por MARCA (usando dim_marca). ¿Qué marca tiene el mejor rating promedio?

In [11]:
# RETO 3: rating promedio por marca
conn = sqlite3.connect("tienda_online.db")
q = """SELECT m.brand AS marca, ROUND(AVG(f.rating), 2) AS rating, COUNT(*) AS n
       FROM fact_productos f JOIN dim_marca m ON f.marca_id = m.marca_id
       GROUP BY m.brand ORDER BY rating DESC"""
print(pd.read_sql(q, conn).to_string(index=False))
conn.close()

           marca  rating  n
           Knoll    4.88  1
    Velvet Touch    4.64  1
    Calvin Klein    4.37  1
  Chic Cosmetics    4.36  1
Annibale Colombo    4.34  2
    Nail Couture    4.32  1
          Chanel    4.26  1
 Dolce & Gabbana    3.96  1
            Dior    3.80  1
     Bath Trends    3.59  1
   Furniture Co.    2.87  1
  Glamour Beauty    2.86  1
           Gucci    2.74  1
         Essence    2.56  1


> **24.** ¿Qué marca tiene el mejor rating? ¿Coincide con lo que esperabas?

**Respuesta:** Knoll, con 4.88, pero es un solo producto; Annibale Colombo, la única marca con 2 productos, promedia 4.34. No era lo que esperaba: marcas de lujo como Gucci (2.74) y Dior (3.80) quedan abajo. Además, los 27 productos de groceries no tienen marca y quedan fuera del JOIN.

---
# PARTE B — Una base de datos que SE ACTUALIZA

Ahora consultarás **Open Food Facts**: una base de datos colaborativa REAL de productos alimenticios de todo el mundo, que **crece cada minuto** con aportes de usuarios. Es el ejemplo perfecto de una "base viva".

## Paso 5 — Extraer de una base colaborativa viva

In [12]:
# Open Food Facts: base colaborativa que se actualiza constantemente
url_off = "https://world.openfoodfacts.org/api/v2/search"
params = {
    "fields": "product_name,brands,categories,nutriscore_grade,countries",
    "page_size": 50,
    "page": 1
}

try:
    r = requests.get(url_off, params=params, timeout=20,
                     headers={"User-Agent": "OptativaIII-UNE-Educacion"})
    r.raise_for_status()
    off = r.json()
    productos_off = off["products"]
    total_base = off.get("count", "desconocido")
    print("EXTRACT en vivo de Open Food Facts.")
    print("Productos recibidos:", len(productos_off))
    print("TOTAL de productos en la base mundial:", total_base, "(este número CRECE con el tiempo)")
except Exception as e:
    print("La API no respondió (", type(e).__name__, "). Usando datos de respaldo.")
    productos_off = [
        {"product_name":"Nutella","brands":"Ferrero","categories":"Spreads","nutriscore_grade":"e","countries":"France"},
        {"product_name":"Coca-Cola","brands":"Coca-Cola","categories":"Beverages","nutriscore_grade":"e","countries":"World"},
        {"product_name":"Oat Drink","brands":"Oatly","categories":"Plant-based","nutriscore_grade":"b","countries":"Sweden"},
        {"product_name":"Greek Yogurt","brands":"Fage","categories":"Dairy","nutriscore_grade":"a","countries":"Greece"},
    ]
    total_base = "2,900,000+ (aprox.)"

print("\nEjemplo:", productos_off[0])

EXTRACT en vivo de Open Food Facts.
Productos recibidos: 50
TOTAL de productos en la base mundial: 4807242 (este número CRECE con el tiempo)

Ejemplo: {'brands': 'Milky Food Professional', 'categories': 'Cheeses, fr:Fromages blancs natures', 'countries': 'Canada, France, Morocco, United States', 'nutriscore_grade': 'c', 'product_name': 'Fromage Blanc Nature'}


> **25.** ¿Cuántos productos hay en total en la base mundial de Open Food Facts? Este número es enorme y crece cada día. ¿Por qué NO podrías descargar toda la base en un CSV para trabajar?

**Respuesta:** 4,807,242 productos en mi consulta (2026-10-11 14:47). Sería un archivo de varios gigas que tardaría mucho en bajar, y para cuando terminara ya estaría desactualizado.

> **26.** Open Food Facts es alimentada por usuarios de todo el mundo (como Wikipedia, pero de productos). Si consultaras esta base hoy y dentro de una semana, ¿obtendrías exactamente los mismos resultados? ¿Por qué?

**Respuesta:** No. Los usuarios agregan y corrigen productos todo el tiempo. Lo vi en esta misma sesión: el total era 4,807,198 a las 14:47:24 (notebook de investigación) y 4,807,242 a las 14:47:51.

> **27.** ¿Qué reto de ingeniería de datos añade una fuente que CAMBIA vs un CSV fijo? (Piensa en reproducibilidad, en registrar la fecha, en versionar.)

**Respuesta:** La reproducibilidad: la misma consulta da otro resultado otro día. Por eso hay que registrar la fecha de extracción, guardar snapshots versionados y validar cada carga, porque el contenido y hasta el formato pueden cambiar.

## Paso 6 — Demostrar que los datos cambian

In [13]:
from datetime import datetime

# Convertimos a tabla y registramos CUÁNDO se extrajo
df_off = pd.DataFrame(productos_off)
df_off["fecha_consulta"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Datos extraídos a las:", df_off["fecha_consulta"].iloc[0])
print("Si ejecutas esta celda en otro momento, la marca de tiempo cambiará,")
print("y con una base viva, los productos también pueden cambiar.\n")
print(df_off[["product_name", "brands", "nutriscore_grade"]].head())

Datos extraídos a las: 2026-10-11 14:47:51
Si ejecutas esta celda en otro momento, la marca de tiempo cambiará,
y con una base viva, los productos también pueden cambiar.

             product_name                   brands nutriscore_grade
0    Fromage Blanc Nature  Milky Food Professional                c
1                Sidi Ali                 سيدي علي                a
2                   Perly                   Jaouda                a
3  Eau minérale naturelle                 sidi ali                a
4                Sidi Ali                 Sidi Ali                a


> **28.** La columna `fecha_consulta` registra el momento exacto de la extracción. ¿Por qué en un pipeline sobre datos vivos SIEMPRE se registra la fecha de extracción?

**Respuesta:** Porque sin la fecha no se sabe qué versión de los datos se analizó. Con ella se puede explicar por qué dos resultados difieren y comparar extracciones en el tiempo.

> **29.** Si tu profesor y tú ejecutan esta celda al mismo tiempo, ¿podrían obtener datos ligeramente distintos? ¿Qué implica esto para "reproducir" un análisis?

**Respuesta:** Sí, aunque sea por segundos la base pudo recibir cambios. Reproducir el análisis significa repetirlo sobre el mismo snapshot guardado, no volver a consultar la API.

### Reto 4 — Extrae una página distinta

Cambia `page` de 1 a 2 (o a un número al azar) en la consulta a Open Food Facts y vuelve a ejecutar el Paso 5. Compara: ¿son los mismos productos?

In [14]:
# RETO 4: pedir la pagina 2 y compararla con la pagina 1
import time
productos_p2 = []
for intento in range(3):
    try:
        r2 = requests.get(url_off, params=dict(params, page=2), timeout=20,
                          headers={"User-Agent": "OptativaIII-UNE-Educacion"})
        r2.raise_for_status()
        productos_p2 = r2.json()["products"]
        break
    except Exception as e:
        print("Intento", intento + 1, "fallido (", type(e).__name__, ")")
        time.sleep(3)

nombres_p1 = {p.get("product_name") for p in productos_off}
nombres_p2 = {p.get("product_name") for p in productos_p2}
print("Productos pagina 1:", len(productos_off), "| pagina 2:", len(productos_p2))
print("Nombres repetidos entre paginas:", len(nombres_p1 & nombres_p2))
print("Primeros de la pagina 1:", [p.get("product_name") for p in productos_off[:4]])
print("Primeros de la pagina 2:", [p.get("product_name") for p in productos_p2[:4]])

Productos pagina 1: 50 | pagina 2: 50
Nombres repetidos entre paginas: 3
Primeros de la pagina 1: ['Fromage Blanc Nature', 'Sidi Ali', 'Perly', 'Eau minérale naturelle']
Primeros de la pagina 2: ['Fibres', 'Nocciolata', 'Le Lait demi écrémé', "Menguy's Peanut 100%"]


> **30.** ¿Los productos de la página 2 son distintos a los de la página 1? ¿Cómo usa la paginación una API para no enviarte millones de registros de golpe?

**Respuesta:** Sí, son productos distintos: de 50 nombres solo 3 se repiten. La API divide los resultados en páginas de tamaño fijo (page_size) y el cliente pide la página 1, 2, 3... según necesite, en vez de recibir 4.8 millones de registros de golpe.

## Paso 7 — Tu pipeline sobre la base viva

In [15]:
# Pipeline ETL completo sobre Open Food Facts, cargando a SQLite
df_off_clean = df_off.copy()
df_off_clean["nutriscore_grade"] = df_off_clean["nutriscore_grade"].fillna("desconocido").str.upper()

conn = sqlite3.connect("tienda_online.db")
df_off_clean.to_sql("productos_alimenticios", conn, if_exists="replace", index=False)

# Consulta: cuántos productos por calificación nutricional (Nutri-Score)
q = """
SELECT nutriscore_grade AS nutriscore, COUNT(*) AS n
FROM productos_alimenticios
GROUP BY nutriscore_grade
ORDER BY n DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))
conn.close()

    nutriscore  n
             A 20
             E  9
             B  9
             C  6
             D  4
       UNKNOWN  1
NOT-APPLICABLE  1


> **31.** ¿Qué calificación de Nutri-Score (A a E) es la más común entre los productos que extrajiste? (Con datos vivos, puede variar.)

**Respuesta:** A, con 20 de 50 productos; le siguen B y E con 9 cada una. Hubo 2 sin calificación válida (UNKNOWN y NOT-APPLICABLE).

> **32.** Acabas de construir un pipeline que consume una base viva y la carga a tu almacén. ¿Cuál es la ventaja de guardar en SQLite un "corte" (snapshot) de datos que cambian?

**Respuesta:** El snapshot congela los datos de un momento: se pueden reanalizar, auditar o comparar con otro corte aunque la base ya haya cambiado, y sin depender de que la API esté disponible.

---
# PARTE B (cont.) — Modelado y decisiones

## Paso 8 — Estrella vs copo de nieve

In [16]:
print("""
ESTRELLA:                          COPO DE NIEVE:
  dim_categoria --- FACT             dim_categoria -- dim_departamento
                                            |
                                          FACT
Estrella: menos JOINs, más rápido, algo de redundancia.
Copo de nieve: dimensiones divididas, menos redundancia, más JOINs (más lento).
""")


ESTRELLA:                          COPO DE NIEVE:
  dim_categoria --- FACT             dim_categoria -- dim_departamento
                                            |
                                          FACT
Estrella: menos JOINs, más rápido, algo de redundancia.
Copo de nieve: dimensiones divididas, menos redundancia, más JOINs (más lento).



> **33.** ¿Cuál es la diferencia entre estrella y copo de nieve? (Pista: si las dimensiones se subdividen en más tablas.)

**Respuesta:** En la estrella cada dimensión es una sola tabla unida directo a los hechos. En el copo de nieve las dimensiones se normalizan y se dividen en subtablas (categoría → departamento), lo que agrega JOINs.

> **34.** Para un dashboard de la tienda que se consulta miles de veces al día, ¿estrella o copo de nieve? Justifica con el trade-off velocidad vs redundancia.

**Respuesta:** Estrella: con miles de consultas al día importa más la velocidad (menos JOINs) que ahorrar algo de espacio repitiendo datos en las dimensiones.

> **35.** Explica el trade-off normalización vs desnormalización: ¿por qué las bases transaccionales (registrar compras) se normalizan, pero las analíticas (analizar ventas) se desnormalizan?

**Respuesta:** La base transaccional recibe muchas escrituras: normalizar evita duplicados y errores al actualizar, porque cada dato vive en un solo lugar. La analítica casi solo se lee con consultas grandes: desnormalizar reduce JOINs y acelera las lecturas, y la redundancia no estorba porque casi no se actualiza.

### Reto 5 — Pipeline parametrizable para cualquier categoría

Escribe una función `extraer_categoria(cat)` que consulte DummyJSON para una categoría específica (`https://dummyjson.com/products/category/{cat}`) y devuelva un DataFrame limpio. Pruébala con "smartphones" y "laptops".

In [17]:
# RETO 5: funcion parametrizable por categoria
def extraer_categoria(cat):
    url = f"https://dummyjson.com/products/category/{cat}"
    try:
        r = requests.get(url, timeout=15); r.raise_for_status()
        prods = r.json()["products"]
        return pd.DataFrame(prods)[["title", "price", "rating", "stock"]]
    except Exception as e:
        print("Error:", e); return pd.DataFrame()

for cat in ["smartphones", "laptops"]:
    d = extraer_categoria(cat)
    print(f"\n{cat}: {len(d)} productos")
    print(d.to_string(index=False))


smartphones: 16 productos
             title   price  rating  stock
         iPhone 5s  199.99    2.83     25
          iPhone 6  299.99    3.41     60
     iPhone 13 Pro 1099.99    4.12     56
          iPhone X  899.99    2.51     37
          Oppo A57  249.99    3.94     19
 Oppo F19 Pro Plus  399.99    3.51     78
           Oppo K1  299.99    4.25     55
        Realme C35  149.99    4.20     48
          Realme X  299.99    3.70     12
         Realme XT  349.99    4.58     80
 Samsung Galaxy S7  299.99    3.30     67
 Samsung Galaxy S8  499.99    4.40      0
Samsung Galaxy S10  699.99    3.06     19
           Vivo S1  249.99    3.50     50
           Vivo V9  299.99    3.60     82
          Vivo X21  499.99    4.26      7

laptops: 5 productos
                               title   price  rating  stock
Apple MacBook Pro 14 Inch Space Grey 1999.99    3.65     24
 Asus Zenbook Pro Dual Screen Laptop 1799.99    3.95     45
               Huawei Matebook X Pro 1399.99    4.98     

> **36.** ¿Cuántos productos devolvió cada categoría? ¿Por qué una función parametrizable (que recibe la categoría) es más útil que copiar y pegar código para cada una?

**Respuesta:** smartphones devolvió 16 productos y laptops 5. Con la función solo cambia el parámetro: hay un único lugar donde corregir errores, no se duplica código y sirve para cualquier categoría.

## Paso 9 — Validación de calidad del pipeline

In [18]:
conn = sqlite3.connect("tienda_online.db")
datos = pd.read_sql("SELECT * FROM fact_productos", conn)

print("=== CONTROLES DE CALIDAD ===")
print(f"1. ¿Precios negativos? {(datos['price'] < 0).sum()} filas")
print(f"2. ¿Ratings fuera de [0,5]? {((datos['rating']<0)|(datos['rating']>5)).sum()} filas")
print(f"3. ¿Stock negativo? {(datos['stock'] < 0).sum()} filas")
print(f"4. ¿IDs duplicados? {datos['id'].duplicated().sum()} filas")
print(f"5. Total de registros: {len(datos)}")
conn.close()

=== CONTROLES DE CALIDAD ===
1. ¿Precios negativos? 0 filas
2. ¿Ratings fuera de [0,5]? 0 filas
3. ¿Stock negativo? 0 filas
4. ¿IDs duplicados? 0 filas
5. Total de registros: 50


> **37.** ¿El almacén pasó todos los controles? ¿Por qué un pipeline debe VALIDAR y no solo mover datos, especialmente si vienen de una fuente externa que no controlas?

**Respuesta:** Sí: 0 filas en los cuatro controles, con 50 registros. Una fuente externa puede mandar datos erróneos o cambiar sin aviso; si el pipeline solo mueve datos, el error llega al almacén y a los reportes sin que nadie lo note.

> **38.** Los datos vienen de una API externa. ¿Qué pasaría si un día la API cambia el nombre de un campo (ej. "price" → "precio")? ¿Cómo lo detectaría tu pipeline?

**Respuesta:** El filtro de columnas omitiría "price" sin avisar y el pipeline tronaría con KeyError al calcular precio_final. Para detectarlo a tiempo hay que validar el esquema al extraer: comprobar que estén las columnas esperadas y detener la carga con una alerta si falta alguna.

## Paso 10 — Cierre reflexivo

> **39.** Resume el flujo completo: fuente ONLINE → ? → ? → ? → consulta. Nombra cada etapa.

**Respuesta:** Fuente online (API) → Extract (requests + JSON) → Transform (tabla, limpieza, features y fecha) → Load (esquema estrella en SQLite) → validación de calidad → consulta SQL.

> **40.** ¿Qué diferencia hay entre construir un pipeline sobre un CSV fijo (actividad anterior) y sobre una API viva (esta actividad)? Menciona 2 retos nuevos.

**Respuesta:** Un CSV fijo siempre da lo mismo. Con la API: (1) los datos cambian, así que hay que registrar la fecha y guardar snapshots para poder reproducir; (2) la fuente puede fallar o cambiar de formato, así que hay que manejar errores y validar el esquema.

> **41.** Mercado Libre tiene una API, pero desde hace unos años **exige autenticación** (registrar una app y obtener un token OAuth). ¿Por qué crees que una empresa protegería su API con autenticación, mientras que Open Food Facts la deja abierta? (Piensa en el modelo de negocio.)

**Respuesta:** Para Mercado Libre sus datos son un activo del negocio: con autenticación controla quién los usa, limita el abuso y la carga en sus servidores, puede restringir o cobrar el acceso y protege datos de vendedores y compradores. Open Food Facts es un proyecto abierto sin fines de lucro cuya misión es que los datos se usen libremente.

> **42.** En tu carrera de Ingeniería en Computación, ¿en qué proyecto tendrías que consumir una API que se actualiza (clima, precios, redes sociales, transporte)? Describe sus etapas ETL.

**Respuesta:** Una app de transporte público en tiempo real. Extract: consultar cada 30 segundos la API con la posición de cada camión. Transform: limpiar coordenadas, calcular el retraso contra el horario y agregar la fecha. Load: guardar en una base con fact_posiciones + dim_ruta y dim_parada para mostrar tiempos de llegada.

---
# TAREA — Pipeline documentado sobre una fuente viva

Como tarea, construirás un pipeline ETL **documentado** que consuma una API online (DummyJSON, Open Food Facts, u otra pública sin autenticación) y almacene los datos estructurados en SQLite.

### Requisitos del entregable

1. **EXTRACT:** consumir una API pública real, con manejo de errores (try/except).
2. **TRANSFORM:** convertir el JSON en tabla, limpiar, y crear al menos DOS features derivadas. Incluir la **fecha de extracción**.
3. **Modelado:** diseñar un esquema estrella (al menos 1 dimensión + 1 tabla de hechos).
4. **LOAD:** cargar a SQLite.
5. **Validación:** al menos 3 controles de calidad.
6. **Consulta analítica:** al menos 2 consultas SQL con `GROUP BY` y `JOIN`.
7. **Documentación:** comentarios en cada fase + un texto que explique de qué fuente viva extrae y por qué sus datos cambian.

Consulta el notebook de investigación a profundidad (`S6L_investigacion_ALUMNO.ipynb`) para el enfoque de validación crítica y el proyecto final.

## Tarea resuelta — Pipeline documentado sobre Open Food Facts

**Fuente viva:** Open Food Facts, una base colaborativa de alimentos. Usuarios de todo el mundo agregan
productos nuevos y corrigen los existentes (marcas, ingredientes, tablas nutricionales) a cada minuto,
y el Nutri-Score se recalcula cuando cambian esos datos. Por eso dos extracciones en momentos distintos
pueden traer productos o valores diferentes, y el pipeline guarda la fecha de cada extracción.

In [19]:
import time

# ---------- 1. EXTRACT: API publica con manejo de errores y reintentos ----------
URL_OFF = "https://world.openfoodfacts.org/api/v2/search"
PARAMS = {"fields": "code,product_name,brands,nutriscore_grade,nova_group,nutriments",
          "page_size": 100, "page": 1}

def extraer_off(intentos=3):
    """Pide una pagina a Open Food Facts; si la API falla reintenta y al final usa un respaldo."""
    for i in range(1, intentos + 1):
        try:
            r = requests.get(URL_OFF, params=PARAMS, timeout=20,
                             headers={"User-Agent": "OptativaIII-UNE-Educacion"})
            r.raise_for_status()
            return r.json()["products"], "API en vivo"
        except Exception as e:
            print(f"Intento {i} fallido ({type(e).__name__})")
            time.sleep(3)
    respaldo = [{"code": "3017620422003", "product_name": "Nutella", "brands": "Ferrero",
                 "nutriscore_grade": "e", "nova_group": 4,
                 "nutriments": {"energy-kcal_100g": 539, "sugars_100g": 56.3,
                                "fat_100g": 30.9, "salt_100g": 0.107}}]
    return respaldo, "respaldo"

crudos, fuente_tarea = extraer_off()
print("Fuente:", fuente_tarea, "| productos extraidos:", len(crudos))

# ---------- 2. TRANSFORM: JSON anidado -> tabla, limpieza y features ----------
filas = []
for p in crudos:
    n = p.get("nutriments") or {}
    filas.append({
        "codigo": p.get("code"),
        "producto": p.get("product_name") or "sin nombre",
        "marca": (p.get("brands") or "sin marca").split(",")[0].strip(),
        "nutriscore": p.get("nutriscore_grade"),
        "kcal_100g": n.get("energy-kcal_100g"),
        "azucar_100g": n.get("sugars_100g"),
        "grasa_100g": n.get("fat_100g"),
        "sal_100g": n.get("salt_100g"),
    })
alim = pd.DataFrame(filas).drop_duplicates("codigo")

# Limpieza: numeros como numeros y Nutri-Score valido solo de A a E
for col in ["kcal_100g", "azucar_100g", "grasa_100g", "sal_100g"]:
    alim[col] = pd.to_numeric(alim[col], errors="coerce")
alim["nutriscore"] = alim["nutriscore"].str.upper()
alim.loc[~alim["nutriscore"].isin(list("ABCDE")), "nutriscore"] = "Desconocido"

# Feature 1: es_saludable (Nutri-Score A o B)
alim["es_saludable"] = alim["nutriscore"].isin(["A", "B"]).astype(int)
# Feature 2: nivel de azucar con los umbrales del semaforo nutricional (5 g y 22.5 g por 100 g)
alim["nivel_azucar"] = pd.cut(alim["azucar_100g"], bins=[-0.1, 5, 22.5, 100],
                              labels=["Bajo", "Medio", "Alto"]).astype(str)
# Fecha de extraccion: indispensable porque la fuente cambia
alim["fecha_extraccion"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print("Tabla transformada:", alim.shape)

# ---------- 3. MODELADO: esquema estrella (2 dimensiones + 1 tabla de hechos) ----------
dim_nutri = pd.DataFrame({"nutriscore": sorted(alim["nutriscore"].unique())})
dim_nutri["nutri_id"] = range(1, len(dim_nutri) + 1)
dim_marca_off = pd.DataFrame({"marca": alim["marca"].unique()})
dim_marca_off["marca_id"] = range(1, len(dim_marca_off) + 1)
fact_alimentos = (alim.merge(dim_nutri, on="nutriscore").merge(dim_marca_off, on="marca")
                  [["codigo", "producto", "nutri_id", "marca_id", "kcal_100g", "azucar_100g",
                    "grasa_100g", "sal_100g", "es_saludable", "nivel_azucar", "fecha_extraccion"]])

# ---------- 4. LOAD: SQLite ----------
conn = sqlite3.connect("alimentos_vivos.db")
dim_nutri.to_sql("dim_nutriscore", conn, if_exists="replace", index=False)
dim_marca_off.to_sql("dim_marca", conn, if_exists="replace", index=False)
fact_alimentos.to_sql("fact_alimentos", conn, if_exists="replace", index=False)

# ---------- 5. VALIDACION: controles de calidad sobre lo cargado ----------
f = pd.read_sql("SELECT * FROM fact_alimentos", conn)
print("\n=== CONTROLES DE CALIDAD ===")
print("1. kcal fuera de [0, 900]:", ((f["kcal_100g"] < 0) | (f["kcal_100g"] > 900)).sum())
print("2. azucar fuera de [0, 100] g:", ((f["azucar_100g"] < 0) | (f["azucar_100g"] > 100)).sum())
print("3. codigos duplicados:", f["codigo"].duplicated().sum())
print("4. filas sin kcal (dato faltante):", f["kcal_100g"].isna().sum(), "de", len(f))

# ---------- 6. CONSULTAS ANALITICAS (GROUP BY + JOIN) ----------
q1 = """
SELECT d.nutriscore, COUNT(*) AS productos,
       ROUND(AVG(f.kcal_100g), 1) AS kcal_prom, ROUND(AVG(f.azucar_100g), 1) AS azucar_prom
FROM fact_alimentos f JOIN dim_nutriscore d ON f.nutri_id = d.nutri_id
GROUP BY d.nutriscore ORDER BY d.nutriscore"""
q2 = """
SELECT m.marca, COUNT(*) AS productos, ROUND(100.0 * AVG(f.es_saludable), 0) AS pct_saludable
FROM fact_alimentos f JOIN dim_marca m ON f.marca_id = m.marca_id
GROUP BY m.marca ORDER BY productos DESC LIMIT 5"""
print("\nConsulta 1 - perfil nutricional por Nutri-Score:")
print(pd.read_sql(q1, conn).to_string(index=False))
print("\nConsulta 2 - marcas con mas productos y % saludable:")
print(pd.read_sql(q2, conn).to_string(index=False))
conn.close()

Intento 1 fallido (HTTPError)


Intento 2 fallido (HTTPError)


Fuente: API en vivo | productos extraidos: 100
Tabla transformada: (100, 11)

=== CONTROLES DE CALIDAD ===
1. kcal fuera de [0, 900]: 0
2. azucar fuera de [0, 100] g: 0
3. codigos duplicados: 0
4. filas sin kcal (dato faltante): 6 de 100

Consulta 1 - perfil nutricional por Nutri-Score:
 nutriscore  productos  kcal_prom  azucar_prom
          A         34      200.1          2.8
          B         17      186.6          4.6
          C         16      280.6          7.8
          D         11      384.8          5.5
Desconocido          3       91.5          0.0
          E         19      447.6         29.3

Consulta 2 - marcas con mas productos y % saludable:
    marca  productos  pct_saludable
   Jaouda          8           88.0
   Danone          5           40.0
   Gerblé          4           50.0
sin marca          3            0.0
  Nutella          3            0.0


---
## Cierre y entrega

Construiste pipelines ETL que consumen **datos vivos** de la web: una tienda online y una base colaborativa que se actualiza en tiempo real. Aplicaste arquitecturas de datos, modelado dimensional y validación de calidad sobre fuentes que cambian entre una consulta y otra — el escenario real de un ingeniero de datos.

### Entrega en Google Classroom

1. Verifica que **todas las celdas corran sin error** (Entorno de ejecución → Ejecutar todo).
2. Confirma que respondiste las **50 preguntas** y los **5 retos**.
3. Adjunta tu **pipeline documentado** (Tarea).
4. **Archivo → Guardar una copia en Drive** y comparte el enlace en Classroom.

### Rúbrica

| Criterio | Puntos |
|---|---|
| Conceptos de arquitectura (ETL, batch, warehouse) correctos | 15 |
| Retos de código resueltos (5) | 20 |
| Respuestas a las 50 preguntas | 30 |
| **Tarea: pipeline desde API viva, documentado** | 25 |
| Modelado dimensional (esquema estrella) correcto | 10 |
| **Total** | **100** |

---
*Universidad de Especialidades UNE · Plantel Centro · Optativa III: Ciencia de Datos*
*Fuentes vivas: DummyJSON (tienda online) y Open Food Facts (base colaborativa).*